In [ ]:
from pathlib import Path
import subprocess

# 1. Clone repo từ GitHub
REPO = Path("/kaggle/working/Speech-to-Text-PhoWhisper")
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/apase95/Speech-to-Text-PhoWhisper.git", str(REPO)],
        check=True
    )
    print("Đã clone repo thành công.")
else:
    print("Repo đã tồn tại.")

# 2. Cài đặt jiwer (thư viện duy nhất thiếu trên image Kaggle mới)
# Lưu ý: Nếu bạn bắt buộc muốn hạ cấp toàn bộ thư viện theo requirements.txt thì đổi thành:
# !pip install -q -r /kaggle/working/Speech-to-Text-PhoWhisper/kaggle_training/requirements.txt
!pip install -q jiwer

# 3. Kiểm tra nhanh GPU và thư viện
import torch, transformers, jiwer
print(f"Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()} ({torch.cuda.device_count()} GPUs)")
print(f"Transformers: {transformers.__version__}")

In [ ]:
import hashlib
import json
import os
import random
from pathlib import Path
import pandas as pd
import soundfile as sf

# 1. Tìm nhanh metadata_train.csv mà TUYỆT ĐỐI KHÔNG đi vào thư mục wavs/
DATA_ROOT = None
for root, dirs, files in os.walk("/kaggle/input"):
    # Cắt bỏ thư mục wavs khỏi danh sách duyệt để không bị treo 600k files
    dirs[:] = [d for d in dirs if d.lower() not in {"wavs", "wav", "audio", "audios"}]
    if "metadata_train.csv" in files:
        DATA_ROOT = Path(root)
        break

assert DATA_ROOT is not None, "Chưa tìm thấy metadata_train.csv trong /kaggle/input!"
print("Tìm thấy dataset ngay lập tức tại:", DATA_ROOT)

ROOT = Path("/kaggle/working/vietbud200")
MANIFESTS = ROOT / "manifests"
ROOT.mkdir(parents=True, exist_ok=True)
MANIFESTS.mkdir(parents=True, exist_ok=True)

# 2. Đọc bảng metadata_train.csv (chỉ khoảng 52MB, mất ~1-2 giây)
df = pd.read_csv(
    DATA_ROOT / "metadata_train.csv",
    sep="|",
    dtype=str,
    keep_default_na=False,
)

for col in ["audio_file", "text", "speaker_name"]:
    df[col] = df[col].str.strip()

eligible = df[df["audio_file"].ne("") & df["text"].ne("")].drop_duplicates(subset=["audio_file"]).copy()
audio_ids = pd.to_numeric(
    eligible["audio_file"].str.extract(r"(?:^|/)audio(\d+)\.wav$", expand=False),
    errors="coerce"
)
eligible = eligible.loc[audio_ids.gt(1250)]

# 3. Lấy ngẫu nhiên 200 mẫu (chỉ kiểm tra trực tiếp từng file một, không quét thư mục)
selected_rows = []
for _, row in eligible.sample(frac=1, random_state=42).iterrows():
    audio_path = DATA_ROOT / row["audio_file"]
    if audio_path.is_file():
        selected_rows.append(row)
        if len(selected_rows) == 200:
            break

assert len(selected_rows) == 200, f"Chỉ tìm được {len(selected_rows)} audio"
pd.DataFrame(selected_rows).to_csv(ROOT / "selected_train_200.csv", sep="|", index=False)

# 4. Kiểm tra 200 file WAV và xuất train.json (180) + validation.json (20)
rows = []
seen_hashes = {}
total_seconds = 0.0

for source in selected_rows:
    relative = Path(source["audio_file"])
    audio_id = relative.name
    path = DATA_ROOT / relative
    text = source["text"]

    info = sf.info(path)
    assert info.frames > 0 and info.samplerate > 0 and 0 < info.duration <= 30, f"Audio lỗi: {audio_id}"

    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest not in seen_hashes, f"Trùng byte audio: {audio_id}"
    seen_hashes[digest] = audio_id
    total_seconds += info.duration

    rows.append({
        "id": audio_id,
        "audio_path": str(path),
        "text": text,
        "speaker": source["speaker_name"],
        "sha256": digest,
    })

rows.sort(key=lambda r: r["id"])
random.Random(42).shuffle(rows)
train_data, val_data = rows[:180], rows[180:]

(MANIFESTS / "train.json").write_text(json.dumps(train_data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(MANIFESTS / "validation.json").write_text(json.dumps(val_data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print(f"HOÀN TẤT: Đã tạo {len(train_data)} train, {len(val_data)} validation ({round(total_seconds/60, 2)} phút).")

In [ ]:
from pathlib import Path

REPO = Path("/kaggle/working/Speech-to-Text-PhoWhisper")
ORIG_SCRIPT = REPO / "kaggle_training" / "train.py"
FP32_SCRIPT = Path("/kaggle/working/vietbud200/train_fp32.py")

assert ORIG_SCRIPT.is_file(), "Chưa thấy repo! Hãy chắc chắn bạn đã chạy Cell 1 (git clone)."

source = ORIG_SCRIPT.read_text(encoding="utf-8")

# Chuyển fp16 sang False để tránh lỗi grad_norm: inf trên Tesla T4
if "fp16=torch.cuda.is_available()" in source:
    patched_source = source.replace("fp16=torch.cuda.is_available()", "fp16=False")
else:
    patched_source = source

FP32_SCRIPT.write_text(patched_source, encoding="utf-8")
print("Đã tạo script huấn luyện FP32 tại:", FP32_SCRIPT)

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path("/kaggle/working/vietbud200")
SCRIPT = ROOT / "train_fp32.py"
MANIFESTS = ROOT / "manifests"
BASELINE_OUT = ROOT / "baseline-fp32"
PILOT_OUT = ROOT / "pilot-fp32-1epoch"
LOG_FILE = ROOT / "training.log"
MODEL_NAME = "vinai/PhoWhisper-base"

# Đổi thành True nếu bạn muốn xóa kết quả cũ để chạy lại từ đầu
FORCE_RERUN = False

if FORCE_RERUN:
    shutil.rmtree(BASELINE_OUT, ignore_errors=True)
    shutil.rmtree(PILOT_OUT, ignore_errors=True)
    print("Đã xóa kết quả cũ để chạy lại từ đầu.\n")

env = os.environ.copy()
for k in list(env):
    if k in {"RANK", "LOCAL_RANK", "WORLD_SIZE", "LOCAL_WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"} or k.startswith(("TORCHELASTIC_", "ACCELERATE_")):
        env.pop(k, None)
env["CUDA_VISIBLE_DEVICES"] = "0"
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
env["TQDM_DISABLE"] = "1"

with LOG_FILE.open("a", encoding="utf-8") as log_f:
    # 1. Đánh giá Baseline
    if not BASELINE_OUT.exists():
        print("1. Đang chạy đánh giá Baseline... (chi tiết ghi vào training.log)", flush=True)
        subprocess.run([
            sys.executable, "-u", str(SCRIPT), "evaluate",
            "--model", MODEL_NAME,
            "--manifest", str(MANIFESTS / "validation.json"),
            "--output", str(BASELINE_OUT),
            "--workers", "0",
        ], env=env, stdout=log_f, stderr=subprocess.STDOUT, check=True)
        print("   -> Hoàn tất Baseline!")
    else:
        print("1. Baseline đã có sẵn kết quả tại:", BASELINE_OUT)

    # 2. Huấn luyện Pilot
    if not PILOT_OUT.exists():
        print("2. Đang huấn luyện Pilot 1 Epoch... (mất khoảng 50s)", flush=True)
        subprocess.run([
            sys.executable, "-u", str(SCRIPT), "train",
            "--model", MODEL_NAME,
            "--manifest", str(MANIFESTS / "train.json"),
            "--validation", str(MANIFESTS / "validation.json"),
            "--output", str(PILOT_OUT),
            "--seed", "42",
            "--learning-rate", "1e-5",
            "--batch", "1",
            "--accumulation", "1",
            "--workers", "0",
            "--max-steps", "180",
            "--save-steps", "60",
        ], env=env, stdout=log_f, stderr=subprocess.STDOUT, check=True)
        print("   -> Hoàn tất huấn luyện Pilot!")
    else:
        print("2. Pilot đã có sẵn kết quả tại:", PILOT_OUT)

In [ ]:
import json
from pathlib import Path

ROOT = Path("/kaggle/working/vietbud200")
BASELINE_METRICS = ROOT / "baseline-fp32" / "metrics.json"
PILOT_METRICS = ROOT / "pilot-fp32-1epoch" / "metrics.json"
STATE_PATH = ROOT / "pilot-fp32-1epoch" / "trainer_state.json"

if BASELINE_METRICS.is_file() and PILOT_METRICS.is_file():
    b_data = json.loads(BASELINE_METRICS.read_text())
    p_data = json.loads(PILOT_METRICS.read_text())
    
    print(f"{'Chỉ số':<15} | {'Baseline (Gốc)':<18} | {'Sau 1 Epoch Fine-tune':<22}")
    print("-" * 60)
    print(f"{'Loss':<15} | {b_data['test_loss']:<18.4f} | {p_data['test_loss']:<22.4f}")
    print(f"{'WER (%)':<15} | {b_data['test_wer']*100:<18.2f} | {p_data['test_wer']*100:<22.2f}")
    print(f"{'CER (%)':<15} | {b_data['test_cer']*100:<18.2f} | {p_data['test_cer']*100:<22.2f}")

if STATE_PATH.is_file():
    state = json.loads(STATE_PATH.read_text())
    print("\n=== CHI TIẾT VALIDATION THEO TỪNG MỐC STEP ===")
    for entry in state.get("log_history", []):
        if "eval_loss" in entry:
            print(
                f"Step {entry['step']:>3} (Epoch {entry['epoch']:.2f}) -> "
                f"eval_loss: {entry['eval_loss']:.4f} | "
                f"WER: {entry['eval_wer']*100:.2f}% | "
                f"CER: {entry['eval_cer']*100:.2f}%"
            )